# Notebook 05 of 7 — What-If + Attribution + Paper (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1441](https://github.com/prajoria/OpenBB/issues/1441) · Track A counterpart: [`../portfolio/05-whatif-attribution-and-paper.ipynb`](../portfolio/05-whatif-attribution-and-paper.ipynb).

---

## Where we are in Sam's story

NB04 (Track B) surfaced three names using free-authoritative smart-money signals: MSFT (insider cluster-buy + 13F drift up), NVDA (13F net-adds), AMD (insider selling + 13F drift down). My instinct is to add to MSFT + NVDA and close AMD. Before I do any of that with real money — I diff the portfolio, decompose where returns came from, and paper-trade the plan.

By the end we can answer:

> *What would my three intuitive trades actually do to my book — and did the free-only signal shape make me trade for the right reasons?*


## 0. Before we run anything

Same venv rule as every notebook in the series — `.venv_portfolio`. State goes into `.notebook_state/` (gitignored). We pickle to `paper_blotter_free.pkl` — do NOT overwrite Track A's `paper_blotter.pkl`.


In [ ]:
# [Track B / NB05 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB05 looks different from Track A

The good news: the *math* in NB05 doesn't need paid data at all. The what-if diff (§3) is pure weight-space arithmetic on the basket. The Brinson-Fachler decomposition (§4) is a reference implementation shipped in `openbb_portfolio_intel.analytics.brinson.oracle` — no provider call. The paper engine (§5-§6) is `openbb_portfolio_intel.paper` — free, part of the extension.

The one place the tracks diverge is *quotes*. To convert basket weights to share counts (so the diff and paper fills are dollar-sensible), we need a per-symbol price. Track A uses `obb.equity.price.quote(provider="fmp_cached")` — intraday, ~15 minute latency. Track B uses `provider="cboe"` — EOD only, but free-authoritative and richer (33-column shape including bid/ask/last).

Provider chain for this notebook:

| Data path | Provider (Track B) |
|---|---|
| Quote inputs for what-if / paper marks | `cboe` `EquityQuote` (EOD) |
| Load Track B state | `.notebook_state/basket.json`, `xray_free.pkl`, `smart_money_free.pkl` |
| What-if weight-space diff | Pure computation (no provider) |
| Brinson-Fachler synthetic | `openbb_portfolio_intel.analytics.brinson.oracle` |
| Paper trading engine | `openbb_portfolio_intel.paper` |

Latency gap vs Track A: CBOE quotes are EOD-only — so paper fills marked against them reflect yesterday's close, not intraday. For a Monday-morning planning routine that's fine; for intraday execution rehearsal, Track A's fresher fmp_cached path wins.

Bare-term pointers (all cited with Investopedia links in Track A NB05; not re-cited here): position sizing, Kelly criterion, risk management, attribution analysis, asset allocation, security selection, paper trade, market order, limit order, time in force, GTC, realized P&L, unrealized P&L, buying power, margin.


## 1. Load Track B state from NB01/NB03/NB04

Basket from NB01 (shared with Track A). X-ray + smart-money from Track B NB03/NB04 (`xray_free.pkl`, `smart_money_free.pkl`). If any artifact is missing, fall back to the STORY_BIBLE-locked shapes so the notebook runs standalone.


In [ ]:
# [Track B / NB05 §1] Load Track B state — basket.json + *_free.pkl artifacts
import json
import pickle  # noqa: S403 — trusted local artifact under .notebook_state/
from pathlib import Path

state = Path(".notebook_state")
BASKET_LOCKED = [
    {"symbol": "MSFT",  "weight": 0.12},
    {"symbol": "NVDA",  "weight": 0.10},
    {"symbol": "GOOGL", "weight": 0.08},
    {"symbol": "AAPL",  "weight": 0.08},
    {"symbol": "AMD",   "weight": 0.06},
    {"symbol": "QQQ",   "weight": 0.15},
    {"symbol": "VTI",   "weight": 0.20},
    {"symbol": "VNQ",   "weight": 0.08},
    {"symbol": "BND",   "weight": 0.10},
    {"symbol": "GLD",   "weight": 0.03},
]

basket_path = state / "basket.json"
if basket_path.exists():
    basket = json.loads(basket_path.read_text(encoding="utf-8"))
    for row in basket:
        if "ticker" in row and "symbol" not in row:
            row["symbol"] = row.pop("ticker")
    print(f"Loaded basket from {basket_path} (shared with Track A)")
else:
    basket = BASKET_LOCKED
    print("Regenerated basket from STORY_BIBLE locked list")

xray_artifact = None
xray_free = state / "xray_free.pkl"
if xray_free.exists():
    xray_artifact = pickle.loads(xray_free.read_bytes())  # noqa: S301
    eff = xray_artifact.get("effective_positions", {}) if isinstance(xray_artifact, dict) else {}
    print(f"Loaded xray_free.pkl — {len(eff)} effective positions")
else:
    print("xray_free.pkl not found (skipping x-ray delta comparison)")

sm_artifact = None
sm_free = state / "smart_money_free.pkl"
if sm_free.exists():
    sm_artifact = pickle.loads(sm_free.read_bytes())  # noqa: S301
    n = len(sm_artifact.get("by_symbol", {}))
    print(f"Loaded smart_money_free.pkl — {n} scored names (fixture-labelled per STORY_BIBLE §3)")
else:
    print("smart_money_free.pkl not found (falling back to hand-authored trades)")


Loaded basket from .notebook_state\basket.json (shared with Track A)
Loaded xray_free.pkl — 158 effective positions
Loaded smart_money_free.pkl — 3 scored names (fixture-labelled per STORY_BIBLE §3)


## 2. The three candidate trades — in English first

Same discipline as Track A NB05 §2: write out the trade in plain words before touching the diff engine. If I can't say in one sentence *why* I'm placing the trade, I have no business placing it. The `rationale` field is mandatory on every paper order below — the engine enforces it.

Trades derived from Track B NB04 §6 fixture (same shape as Track A):


In [ ]:
# [Track B / NB05 §2] Three candidate trades in English first — same NB04 fixture
def _get(item, key, default=None):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)

top = (sm_artifact.get("top_conviction") if sm_artifact else None) or []
if top:
    label = sm_artifact.get("fixture_label") if sm_artifact.get("is_fixture") else None
    print("Using top_conviction from smart_money_free rollup"
          + (f" [{label}]" if label else " [live]") + ":")
    trades = []
    for item in top[:3]:
        sym = _get(item, "symbol")
        comp = float(_get(item, "composite", 0.0))
        n = int(_get(item, "signal_count", 0))
        if comp > 0:
            action, delta = "buy", 5
        else:
            action, delta = "close", -20
        trades.append({
            "symbol": sym,
            "action": action,
            "delta_shares": delta,
            "rationale": (
                f"smart_money_free composite {comp:+.2f} ({n} signals)"
                + (f" — {label}" if label else "")
            ),
        })
else:
    print("Falling back to hand-authored trades (smart_money_free returned empty):")
    trades = [
        {"symbol": "NVDA", "action": "buy", "delta_shares": 5,
         "rationale": "Add to NVDA — it's up on the year; free-only signals confirm 13F drift up."},
        {"symbol": "VNQ", "action": "trim", "delta_shares": -10,
         "rationale": "Trim VNQ — REITs weak, rate outlook hostile."},
        {"symbol": "AMD", "action": "close", "delta_shares": -20,
         "rationale": "Close AMD — earnings in <7 days, insider selling cluster."},
    ]

print()
print(f"{'Symbol':<8}{'Action':<10}{'ΔShares':>10}   Rationale")
print("-" * 90)
for t in trades:
    print(f"{t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}   {t['rationale']}")


Using top_conviction from smart_money_free rollup [example — signal shape as of 2026-06-30]:

Symbol  Action       ΔShares   Rationale
------------------------------------------------------------------------------------------
MSFT    buy                5   smart_money_free composite +0.72 (6 signals) — example — signal shape as of 2026-06-30
NVDA    buy                5   smart_money_free composite +0.61 (5 signals) — example — signal shape as of 2026-06-30
AMD     close            -20   smart_money_free composite -0.58 (4 signals) — example — signal shape as of 2026-06-30


## 3. What-if diff — weight-space arithmetic (free-only)

Same computation as Track A NB05 §3 — pure math on the basket weights. Convert current basket weights to nominal share counts using CBOE EOD quotes, apply the trade deltas, recompute weights, print the HHI + effective-N delta.

If the diff shows concentration going *up* when the intuitive read was "this trims risk," that's the tool catching me.


In [ ]:
# [Track B / NB05 §3] What-if weight-space diff — CBOE quotes, in-notebook math
from decimal import Decimal
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

ACCOUNT_SIZE = Decimal("100000")
prices = {}
cboe_sample = None
for p in basket:
    try:
        r = obb.equity.price.quote(symbol=p["symbol"], provider="cboe")
        row = r.results[0]
        last = getattr(row, "last_price", None)
        prices[p["symbol"]] = float(last) if last is not None else 100.0
        if cboe_sample is None and p["symbol"] == "MSFT":
            cboe_sample = row
    except Exception as exc:
        print(f"  {p['symbol']}: CBOE quote failed — {type(exc).__name__} — fallback $100")
        prices[p["symbol"]] = 100.0

if cboe_sample is not None:
    bid = getattr(cboe_sample, "bid", None)
    ask = getattr(cboe_sample, "ask", None)
    ts = getattr(cboe_sample, "last_timestamp", None)
    print(f"CBOE quote sample (MSFT): last={prices.get('MSFT')} bid={bid} ask={ask} ts={ts}")
    print(f"  (EOD-only — for intraday marks Track A's fmp gets fresher data.)")
    print()

current_shares = {
    p["symbol"]: (float(ACCOUNT_SIZE) * p["weight"]) / prices[p["symbol"]]
    for p in basket
}
new_shares = dict(current_shares)
for t in trades:
    new_shares[t["symbol"]] = new_shares.get(t["symbol"], 0.0) + t["delta_shares"]

def _weights(shares_dict, price_dict):
    values = {s: shares_dict[s] * price_dict.get(s, 100.0) for s in shares_dict}
    total = sum(values.values())
    return {s: v/total for s, v in values.items() if total > 0}

w_current = _weights(current_shares, prices)
w_new = _weights(new_shares, prices)

print(f"{'Symbol':<8}{'Before':>10}{'After':>10}{'Δweight':>10}")
print("-" * 40)
for s in sorted(set(w_current) | set(w_new),
                key=lambda x: -abs(w_new.get(x, 0) - w_current.get(x, 0))):
    b, a = w_current.get(s, 0), w_new.get(s, 0)
    d = a - b
    if abs(d) < 0.001: continue
    print(f"{s:<8}{b*100:>9.2f}%{a*100:>9.2f}%{d*100:>+9.2f}%")

def _hhi(w): return sum(x*x for x in w.values())
h_c, h_n = _hhi(w_current), _hhi(w_new)
neff_c = 1/h_c if h_c > 0 else float("nan")
neff_n = 1/h_n if h_n > 0 else float("nan")
print()
print(f"HHI:         {h_c:.4f} → {h_n:.4f}   Δ={h_n-h_c:+.4f}")
print(f"Effective-N: {neff_c:.2f}   → {neff_n:.2f}   Δ={neff_n-neff_c:+.2f}")
if h_n - h_c > 0.001:
    print("\n  → concentration UP: the intuitive trade made things worse (tool caught me).")
elif h_n - h_c < -0.001:
    print("\n  → concentration DOWN: trades achieved the diversification I intended.")


CBOE quote sample (MSFT): last=381.4 bid=381.35 ask=381.4 ts=2026-07-24 15:59:59
  (EOD-only — for intraday marks Track A's fmp gets fresher data.)

Symbol      Before     After   Δweight
----------------------------------------
AMD          6.00%    -4.79%   -10.79%
MSFT        12.00%    15.03%    +3.03%
NVDA        10.00%    11.93%    +1.93%
VTI         20.00%    21.62%    +1.62%
QQQ         15.00%    16.21%    +1.21%
BND         10.00%    10.81%    +0.81%
AAPL         8.00%     8.65%    +0.65%
GOOGL        8.00%     8.65%    +0.65%
VNQ          8.00%     8.65%    +0.65%
GLD          3.00%     3.24%    +0.24%

HHI:         0.1206 → 0.1473   Δ=+0.0267
Effective-N: 8.29   → 6.79   Δ=-1.50

  → concentration UP: the intuitive trade made things worse (tool caught me).


## 4. Brinson-Fachler attribution — synthetic reference (free-only)

Same reference implementation as Track A NB05 §5 — `openbb_portfolio_intel.analytics.brinson.oracle.brinson_reference`. Free, part of the extension. Decomposes active return into allocation vs selection vs interaction.

For most retail books, **allocation dominates** — sector-timing matters more than name-picking inside a sector. If your alpha is negative and allocation is the main contributor, you were betting on sector-timing without knowing it.


In [ ]:
# [Track B / NB05 §4] Brinson-Fachler synthetic — same reference impl as Track A
import pandas as pd
from openbb_portfolio_intel.analytics.brinson.oracle import brinson_reference

df = pd.DataFrame([
    {"sector":"Technology",    "w_p":0.50, "w_b":0.30, "r_p": 0.08, "r_b": 0.10},
    {"sector":"Financials",    "w_p":0.05, "w_b":0.15, "r_p": 0.02, "r_b": 0.05},
    {"sector":"Healthcare",    "w_p":0.05, "w_b":0.15, "r_p": 0.03, "r_b": 0.02},
    {"sector":"Consumer",      "w_p":0.10, "w_b":0.15, "r_p": 0.04, "r_b": 0.06},
    {"sector":"Real Estate",   "w_p":0.10, "w_b":0.05, "r_p":-0.02, "r_b":-0.01},
    {"sector":"Bond Fund",     "w_p":0.10, "w_b":0.10, "r_p": 0.01, "r_b": 0.02},
    {"sector":"Commodity",     "w_p":0.10, "w_b":0.10, "r_p": 0.05, "r_b": 0.03},
])

effects = brinson_reference(df)
print("Brinson-Fachler attribution (portfolio vs benchmark, synthetic):")
print(f"  active_return:       {effects.active_return:+.4f}")
print(f"  allocation:          {effects.allocation:+.4f}")
print(f"  selection:           {effects.selection:+.4f}")
print(f"  interaction:         {effects.interaction:+.4f}")
print()
sum_effects = effects.allocation + effects.selection + effects.interaction
print(f"Reconciliation:  alloc+select+interact = {sum_effects:+.4f}  vs active_return {effects.active_return:+.4f}")
print()
if abs(effects.allocation) > abs(effects.selection):
    print(f"  → allocation dominates ({effects.allocation:+.4f} vs {effects.selection:+.4f})")
    print(f"    Sector-timing drove the active return, not stock-picking.")
else:
    print(f"  → selection dominates ({effects.selection:+.4f} vs {effects.allocation:+.4f})")
    print(f"    Stock-picking drove the active return, not sector-timing.")


Brinson-Fachler attribution (portfolio vs benchmark, synthetic):
  active_return:       -0.0035
  allocation:          +0.0095
  selection:           -0.0115
  interaction:         -0.0015

Reconciliation:  alloc+select+interact = -0.0035  vs active_return -0.0035

  → selection dominates (-0.0115 vs +0.0095)
    Stock-picking drove the active return, not sector-timing.


## 5. Paper blotter — placing trades without capital (CBOE-marked)

Same paper engine as Track A — `openbb_portfolio_intel.paper`. Free, part of the extension. The only Track B swap: the `QuoteFetcher` wraps `obb.equity.price.quote(provider="cboe")` instead of `fmp_cached`.

API-shape reminders (already documented in prior Track A commits): `InMemoryAccountStore.create(user_id=..., config=..., now=..., account_id=...)` is the factory; positions are seeded via `position_store.put(account_id, lot, user_id=...)`; over-sized orders return `SubmitResult(status=REJECTED)` rather than raise `OrderRejected`.


In [ ]:
# [Track B / NB05 §5] Paper blotter — CBOE-backed QuoteFetcher
from decimal import Decimal
from datetime import datetime, timezone
from openbb import obb
from openbb_portfolio_intel.paper.accounts import (
    AccountConfig, InMemoryAccountStore,
)
from openbb_portfolio_intel.paper.fills import (
    OrderRequest, OrderType, TimeInForce, Quote, Lot,
    InMemoryPositionStore, submit_order, OrderRejected,
)

account_store = InMemoryAccountStore()
position_store = InMemoryPositionStore()

now = datetime.now(timezone.utc)
cfg = AccountConfig(starting_cash=Decimal("100000"), display_name="Sam-NB05-Free")
account = account_store.create(
    user_id="sam", config=cfg, now=now, account_id="nb05-free-demo",
)

class CboeQuoteFetcher:
    """QuoteFetcher wrapping CBOE EOD quotes — free-authoritative."""
    def __init__(self, price_cache):
        self._cache = price_cache
    def fetch(self, symbol, *, now):
        # Prefer the pre-fetched price_cache from §3 to avoid re-hitting CBOE
        # per order; fall back to live cboe if symbol wasn't in the basket.
        if symbol in self._cache:
            return Quote(symbol=symbol, last=Decimal(str(self._cache[symbol])), quoted_at=now)
        r = obb.equity.price.quote(symbol=symbol, provider="cboe")
        last = r.results[0].last_price or 100.0
        return Quote(symbol=symbol, last=Decimal(str(last)), quoted_at=now)

quote_fetcher = CboeQuoteFetcher(prices)

# Seed inventory so trim/close trades resolve against real lots
seed_lots = [
    Lot(symbol="VNQ", qty=Decimal("40"), avg_cost=Decimal("85"),
        realized_pnl=Decimal("0")),
    Lot(symbol="AMD", qty=Decimal("30"), avg_cost=Decimal("135"),
        realized_pnl=Decimal("0")),
]
for lot in seed_lots:
    position_store.put("nb05-free-demo", lot, user_id="sam")

results = []
for t in trades:
    req = OrderRequest(
        symbol=t["symbol"],
        qty=Decimal(str(t["delta_shares"])),
        order_type=OrderType.MARKET,
        time_in_force=TimeInForce.DAY,
    )
    try:
        r = submit_order(
            req=req, user_id="sam", account_id="nb05-free-demo",
            account_store=account_store, position_store=position_store,
            quote_fetcher=quote_fetcher, now=now,
        )
        results.append((t, r))
    except OrderRejected as exc:
        results.append((t, ("REJECTED", str(exc))))

print("Paper blotter — 3 trades submitted at market (CBOE-marked):")
print(f"  {'Symbol':<8}{'Action':<10}{'ΔShares':>10}{'Fill':>14}   Status")
print(f"  {'-'*8}{'-'*10}{'-'*10}{'-'*14}   {'-'*20}")
for t, r in results:
    if isinstance(r, tuple) and r[0] == "REJECTED":
        print(f"  {t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}{'—':>14}   REJECTED: {r[1][:40]}")
    else:
        fill = getattr(r, "fill", None) or (r.fills[0] if getattr(r, "fills", None) else None)
        status = getattr(r, "status", "?")
        price = getattr(fill, "price", "?") if fill else "?"
        price_str = f"${float(price):,.2f}" if isinstance(price, Decimal) else str(price)
        print(f"  {t['symbol']:<8}{t['action']:<10}{t['delta_shares']:>10}{price_str:>14}   {status}")

post_account = account_store.get("nb05-free-demo", user_id="sam")
print(f"\nPost-trade cash: ${post_account.cash_balance:,.2f} (started at $100,000)")

seen_syms = {t["symbol"] for t in trades}
positions_snapshot = {}
print("\nPost-trade positions:")
for sym in sorted(seen_syms):
    try:
        pos = position_store.get("nb05-free-demo", sym, user_id="sam")
        if pos is not None:
            qty = getattr(pos, "qty", None) or getattr(pos, "quantity", None)
            positions_snapshot[sym] = float(qty) if qty is not None else 0.0
            print(f"  {sym}: qty={qty}")
    except Exception as exc:
        print(f"  {sym}: (store.get raised {type(exc).__name__})")


Paper blotter — 3 trades submitted at market (CBOE-marked):
  Symbol  Action       ΔShares          Fill   Status
  ------------------------------------------   --------------------
  MSFT    buy                5       $381.59   OrderStatus.FILLED
  NVDA    buy                5       $206.90   OrderStatus.FILLED
  AMD     close            -20       $521.25   OrderStatus.FILLED

Post-trade cash: $107,482.52 (started at $100,000)

Post-trade positions:
  AMD: qty=10
  MSFT: qty=5
  NVDA: qty=5


## 6. Low-BP alert — deliberately trigger one

Same shape as Track A NB05 §7 — submit a deliberately over-sized order and confirm the paper engine emits `SubmitResult(status=REJECTED)` with an insufficient-cash reason. This is the shape a real low-buying-power alert would carry.


In [ ]:
# [Track B / NB05 §6] Low-BP alert — deliberately trigger SubmitResult(REJECTED)
from decimal import Decimal
from openbb_portfolio_intel.paper.fills import (
    OrderRequest, OrderRejected, OrderType, OrderStatus,
)

huge = OrderRequest(
    symbol="MSFT",
    qty=Decimal("10000"),  # ~$3.8M of MSFT on a $100k account
    order_type=OrderType.MARKET,
)
alert_fired = False
try:
    r = submit_order(
        req=huge, user_id="sam", account_id="nb05-free-demo",
        account_store=account_store, position_store=position_store,
        quote_fetcher=quote_fetcher, now=now,
    )
    if getattr(r, "status", None) == OrderStatus.REJECTED:
        alert_fired = True
        reason = getattr(r, "reason", "unknown")
        print("Alert fired — SubmitResult(status=REJECTED):")
        print(f"  reason: {reason}")
    else:
        print(f"UNEXPECTED: over-sized order not rejected — result: {r}")
except OrderRejected as exc:
    alert_fired = True
    print("Alert fired — OrderRejected raised:")
    print(f"  reason: {exc}")

if alert_fired:
    print()
    print("Shape a real Alert object would carry:")
    print(f"  Alert(type='low_buying_power', symbol='MSFT', requested_qty=10000,")
    print(f"        cash_available=${post_account.cash_balance:,.2f})")


Alert fired — SubmitResult(status=REJECTED):
  reason: insufficient cash: cash delta -3815907.00000 would drive 'nb05-free-demo' balance to -3708424.47510475, below min_balance=0

Shape a real Alert object would carry:
  Alert(type='low_buying_power', symbol='MSFT', requested_qty=10000,
        cash_available=$107,482.52)


## 7. Save state for NB07 (Track B)

Pickle the blotter state to `.notebook_state/paper_blotter_free.pkl`. **Do NOT overwrite Track A's `paper_blotter.pkl`** — NB07 Track B will read the `_free` variant.


In [ ]:
# [Track B / NB05 §7] Save paper-blotter state — write ONLY to *_free.pkl
import pickle  # noqa: S403 — trusted local artifact under .notebook_state/
from pathlib import Path

state = Path(".notebook_state")
state.mkdir(exist_ok=True)

blotter_artifact = {
    "track": "B (free-only)",
    "trades_submitted": [t for t, _ in results],
    "starting_cash": 100000.0,
    "ending_cash": float(post_account.cash_balance),
    "positions_post": positions_snapshot,
    "quote_source": "cboe (EOD)",
    "whatif_diff": {
        "hhi_before": h_c, "hhi_after": h_n, "hhi_delta": h_n - h_c,
        "neff_before": neff_c, "neff_after": neff_n, "neff_delta": neff_n - neff_c,
    },
    "brinson_effects": {
        "active_return": float(effects.active_return),
        "allocation":    float(effects.allocation),
        "selection":     float(effects.selection),
        "interaction":   float(effects.interaction),
    },
    "gap_vs_track_a": "CBOE quotes are EOD-only; intraday marks require Track A.",
}

out = state / "paper_blotter_free.pkl"
out.write_bytes(pickle.dumps(blotter_artifact))
print(f"Wrote (repo-rel): {str(out):<45}  {out.stat().st_size:,} bytes")
print("Track A's paper_blotter.pkl NOT modified by this notebook.")


Wrote (repo-rel): .notebook_state\paper_blotter_free.pkl         929 bytes
Track A's paper_blotter.pkl NOT modified by this notebook.


---

## What is NOT in this notebook

Same gaps as Track A NB05 plus the free-tier delta:

- **Live broker adapter.** Paper only. The `execution/` module has the   shape a real broker adapter would slot into; not shipped.
- **Margin-call simulation.** Single-account, no margin math yet.
- **Overnight risk report.** The alert engine covers today; overnight-gap   risk projection is future work.
- **Intraday quote marks.** CBOE quote latency is EOD only; for intraday   paper fills the paid Track A path (`fmp_cached`) gets fresher data. If   you're rehearsing intraday execution, use Track A; for a   Monday-morning planning routine, EOD is enough.

## Preview of NB06 (Track B)

The paper trade is on. But NB05 was one moment — one basket, one set of signals, one diff. The strategy behind it ("rebalance to top-K smart-money composite inside the basket every month") hasn't been tested against history. NB06 turns it into a `BacktestConfig` and sees whether the underlying idea has real edge, or whether NB05 was just one lucky what-if.

## 📚 Further reading

Every Investopedia link cited in Track A NB05 (position sizing, Kelly, risk management, attribution, asset allocation, security selection, paper trade, market/limit orders, TIF, GTC, realized/unrealized P&L, buying power, margin) applies unchanged. Not re-cited here.

**Canonical references** (unchanged from Track A):

- Brinson, G. P., Hood, L. R. & Beebower, G. L. — "Determinants of   Portfolio Performance," *Financial Analysts Journal* 42(4), 1986. The   original attribution decomposition; the Brinson-Fachler variant used   in §4 extends it by benchmarking allocation against the sector's   excess return.
- Thorp, E. O. — "The Kelly Criterion in Blackjack, Sports Betting, and   the Stock Market," *Handbook of Asset and Liability Management*,   vol. 1, 2006. The practitioner's tour of Kelly sizing including the   arguments for fractional-Kelly ("Kelly-lite") in real-world portfolios.

**Free-authoritative sources used:**

- **CBOE `EquityQuote`** — EOD last/bid/ask for basket-mark quotes   (§3, §5). Not exchange-real-time, but free-authoritative.
- **`openbb_portfolio_intel.analytics.brinson`** — reference   attribution engine (§4).
- **`openbb_portfolio_intel.paper`** — paper trading engine (§5, §6).
